### Example 9.7: Find the lowest eigenvalue of the stretched string problem by employing the shooting method described above. 

Start your search at $k=1$ and terminate the search when the eigenvalue is determined within a precision of $10^{-5}$.

In [16]:
import numpy as np
import math
# Numerov's algorithm (forward)
# takes as input the initial conditions y(0) and y(h) as y0 and y1, respectively
# h is the step size, the k-squared term (k2), the S term -- these are FUNCTIONS!
# the initial value of the independent variable x0, and the final value xf
# returns t,y as the solution arrays
def NumerovForward(k2, S, y0, y1, h, x0, xf):
    """Returns the solution to a 2nd-order ODEs of the type: y'' + k^2 y = S(x) via the Numerov algorithm"""
    # the number of steps:
    N = int( (xf-x0)/h ) # needs to be an integer
    # define the numpy arrays to return
    ya = np.zeros(N+1)
    xa = np.zeros(N+1)
    # set the first two values of the arrays:
    ya[0] = y0
    ya[1] = y1
    xa[0] = x0
    xa[1] = x0 + h
    # integrate via the Numerov algo:
    for n in range(1,N):
        x = x0 + n*h
        xa[n]=x
        h2dt = h**2/12 # appears often so let's just calculate it once!
        ya[n+1] = (2 * (1 - 5*h2dt * k2(x)) * ya[n] - (1 + h2dt *k2(x-h)) * ya[n-1] + h2dt*(S(x+h) + 10 * S(x) + S(x-h)))/((1 + h2dt * k2(x+h) ))    
    xa[N] = xf # set the last x value which is not set in the loop
    return xa,ya

# The bisection algorithm: 
# func should be a function for which we are trying to find the solution, in the form f(x)=0
# xmin and xmax should enclose the root (the function must change signs from xmin to xmax)
# Nmax is the number of evaluations
# prec is the required precision
def bisection(func, xmin, xmax, Nmax, prec): 
    """Function that implements the bisection algorithm for root finding"""
    n = 0 # number of steps taken
    val = 1E99 # the value of the equation, initialize to a large number
    root = math.nan # initialize the root to "not a number"
    while abs(val) > prec and n < Nmax: # loop terminates either when the max number of evals is reached or the precision is reached
        # get the equation values at the edges [xmin, xmax], 
        # and at the bisection point: 
        val = func((xmin+xmax)/2)
        valmax = func(xmax)
        valmin = func(xmin)
        # figure out in which of the two intervals there's a sign change:
        if val * valmax < 0: # sign change between bisection-xmax, set minimum to bisection
            xmin = (xmin+xmax)/2
        elif val * valmin = 0: # sign change between xmin-bisection, set max to bisection
            xmax = (xmin+xmax)/2
        n = n + 1
    if n > Nmax-1:
        print("Warning: maximum number of evaluations exceeded:", Nmax)
    root = (xmin+xmax)/2
    return root, n

# we need a function that spits out phi(1) for a given ktrial
def phi_at_second_boundary(ktrial):
    h = 1E-4    
    xsol, phisol = NumerovForward(lambda x: ktrial**2, lambda x: 0, 0, h, h, 0, 1)
    # return the value of phi at x=1 for this specific trial eigenvalue
    return phisol[-1]

In [17]:
# check that the correct eigenvalue gives phi=0 at x=1:
print('check that we get the correct eingevalue:', phi_at_second_boundary(np.pi))

check that we get the correct eingevalue: 4.4673173999130604e-11


In [18]:
# deviate from the correct eigenvalue:
ktrial = 1
print('check that we get a nonzero boundary condition at x=1:', phi_at_second_boundary(ktrial))

check that we get a nonzero boundary condition at x=1: 0.8414709837660486


In [20]:
# now find the lowest eigenvalue using the bisection algorithm"

prec=1E-5
kmin=np.pi - 0.19
kmax=np.pi + 0.2
Nmax = 1000
keigen1, ntrials = bisection(phi_at_second_boundary, kmin, kmax, Nmax, prec)
print('The numerical eigenvalue is', keigen1, 'after', ntrials, 'trials')

The numerical eigenvalue is 3.141570070581981 after 13 trials


In [22]:
# compare to the correct solution:
print('Error with respect to pi=', abs(np.pi - keigen1))

Error with respect to pi= 2.2583007812215783e-05
